In [ ]:
import glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
import torch.optim.lr_scheduler as lr_scheduler

In [ ]:
dataset_path = "/kaggle/input/datasets/niklaspi1/chess-bitboards/preprocessed_data/*.pt"
chunk_files = glob.glob(dataset_path)
chunk_files.sort()
len(chunk_files)

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
print(device)

In [ ]:
#model
class ChessTransformer(nn.Module):
    def __init__(self, d_model, nhead, num_layers, d_ff, dropout):
        super(ChessTransformer, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=19, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.ReLU()
        )

        self.rank_embedding = nn.Parameter(torch.randn(1, 8, d_model//2))
        self.file_embedding = nn.Parameter(torch.randn(1, 8, d_model//2))

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        self.evaluation_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )


        self.move_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Linear(d_model, 64)
        )


    def forward(self, x, legal_move_mask=None):
        batch_size = x.shape[0]
        
        x = torch.transpose(x, 1, 2)
        x = x.view(batch_size, 19, 8, 8)

        x = self.conv_layers(x)

        x = x.view(batch_size, 512, -1)
        x = torch.transpose(x, 1, 2)

        x = x+torch.cat((self.rank_embedding.repeat_interleave(8, dim=1), self.file_embedding.repeat(1, 8, 1)), 2)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        logits = self.encoder(x)

        evaluation_values = logits[:, 0, :]
        evaluation_logits = self.evaluation_head(evaluation_values)

        move_values = logits[:, 1:, :]
        move_logits = self.move_head(move_values)

        if legal_move_mask is not None:
            move_logits = move_logits.view(batch_size, -1)

            move_logits += legal_move_mask
            
        return evaluation_logits, move_logits

In [ ]:
model = ChessTransformer(
    d_model=512,
    nhead=16,
    num_layers=8,
    d_ff=1024,
    dropout=0.1
).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print("Multiple GPUs")
    
print(model)

num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print(f"Number of parameters: {num_params}")

In [ ]:
def idx_to_array(idx_array, batch_size):
    
    idx_array = idx_array.view(batch_size, -1)
    if idx_array.shape[1] == 1:
        array = torch.zeros(batch_size, 64*64)
    else:
        array = torch.full((batch_size, 64*64), -1e4)
        
    idx_count = idx_array.shape[1]

    valid_mask = idx_array != -1

    rows = torch.arange(batch_size).view(-1, 1).expand(batch_size, idx_count)

    valid_rows = rows[valid_mask]
    valid_cols = idx_array[valid_mask]

    if idx_array.shape[1] == 1:
        array[valid_rows, valid_cols] = 1.0
    else:
        array[valid_rows, valid_cols] = 0.0
        
    return array

In [ ]:
def save_checkpoint(state, filename="chessformer_checkpoint.pth"):
    torch.save(state, filename)
    print(f"Checkpoint saved to {filename}")

def load_checkpoint(filename, model, optimizer, scheduler, scaler):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    return checkpoint['epoch'], checkpoint['file_idx']

In [ ]:
epochs = 20
batch_size = 1024
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

total_steps = (10010000//batch_size) * epochs
scheduler = lr_scheduler.OneCycleLR(
    optimizer, max_lr=5e-4, total_steps=total_steps, pct_start=0.1
)

scaler = GradScaler(device=device)

In [ ]:
start_epoch = 0
start_file_idx = 0
checkpoint_path = "/kaggle/input/datasets/niklaspi1/checkpoint2/chessformer_checkpoint-2.pth"

import os 
if os.path.exists(checkpoint_path):
    start_epoch, start_file_idx = load_checkpoint(
        checkpoint_path, model, optimizer, scheduler, scaler
    )
    start_file_idx += 1
    print(f"Resuming from epoch {start_epoch + 1}, File {start_file_idx}")

for epoch in range(start_epoch, epochs):
    print(f"Start epoch: {epoch+1}!")
    model.train()

    current_start_file = start_file_idx if epoch == start_epoch else 0

    total_loss_eval = 0
    total_loss_move = 0
    total_loss = 0
    
    for file_idx in range(current_start_file, len(chunk_files)):
        file_path = chunk_files[file_idx]

        chunk_data = torch.load(file_path, map_location="cpu")

        dataset = TensorDataset(chunk_data["x"], chunk_data["y"], chunk_data["best"], chunk_data["legal"])

        train_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            drop_last=True
        )


        for batch, (input, eval, best_move, legal_moves) in enumerate(train_loader):
            
            input = input.float().to(device)
            label = eval.float().to(device).unsqueeze(1)
            best_move = best_move.to(device)
            legal_moves_mask = idx_to_array(legal_moves.int(), batch_size).float().to(device)


            optimizer.zero_grad()

            with autocast(device_type=device.type, dtype=torch.float16):
                evaluation, best_moves = model(input, legal_moves_mask)

                loss_eval = F.binary_cross_entropy_with_logits(evaluation, label)
                loss_move = F.cross_entropy(best_moves, best_move.long())
                
                loss = 5 * loss_eval + loss_move

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss_eval += loss_eval.item()
            total_loss_move += loss_move.item()
            total_loss += loss.item()
            
        if (file_idx + 1) % 10 == 0:
            average_loss_eval = total_loss_eval / (1000000 / batch_size)
            average_loss_move = total_loss_move / (1000000 / batch_size)
            average_loss = total_loss / (1000000 / batch_size)
            
            total_loss_eval = 0
            total_loss_move = 0
            total_loss = 0
            
            print(f"Epoch {epoch+1} | Step {file_idx+1} | Evaluation Loss: {average_loss_eval:.4f} | Best Move Loss: {average_loss_move:.4f} | Loss: {average_loss:.4f}")
            torch.save(model.state_dict(), "chessformer_model_weights.pth")

            checkpoint = {
                'epoch': epoch,
                'file_idx': file_idx,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
            }
            save_checkpoint(checkpoint)